# Exercise 6 Capstone: Build Your Own Memory-Enabled Agent

**Estimated Duration:** 75 minutes

In this notebook you will build a complete, working memory-enabled agent from scratch — choosing your own scenario, configuring memory tuning parameters, writing your own conversation turns, and wiring an explicit memory retrieval tool.

Everything you need is already installed. You are not writing library code — you are writing the *application layer* that sits on top of `AgentMemory`.

---

## Before you start — choose your scenario

Pick **one** of the three scenarios below. Your choice determines the `USER_ID`, the `AGENT_PERSONA`, and the kinds of facts your agent should remember. You will fill in the `# YOUR CHOICE:` cells throughout this notebook.

| # | Scenario | What the agent does | What it must remember |
|---|---|---|---|
| A | **IT Help Desk** | Helps a user troubleshoot technical problems | Device type, OS, past issues, resolved tickets — so it never re-diagnoses something already fixed |
| B | **Travel Planning** | Helps a user plan trips | Preferred airlines, hotel tier, dietary restrictions, past destinations — so recommendations feel personal |
| C | **Learning Coach** | Helps a user study a topic | Topics already covered, quiz scores, learning style, current goals — so sessions build on each other |

Write your choice (A, B, or C) here: **YOUR CHOICE: ___**

---

## Cell 1 — Setup: Imports, Environment, and Project Root

Run this cell first. It loads your `.env` file and makes the `memory` package importable.

**You do not need to change anything in this cell.**

In [ ]:
import asyncio
import os
import sys
from pathlib import Path

# ── find project root and add to path ──────────────────────────────────────
def find_project_root(start: Path) -> Path:
    for parent in [start] + list(start.parents):
        if (parent / 'pyproject.toml').exists():
            return parent
    return start

project_root = find_project_root(Path.cwd())
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
print(f'Project root: {project_root}')

# ── load .env ──────────────────────────────────────────────────────────────
try:
    from dotenv import load_dotenv
    load_dotenv(project_root / '.env')
    print('✅ .env loaded')
except ImportError:
    print('⚠️  python-dotenv not found — ensure environment variables are set manually')

# ── validate required env vars ─────────────────────────────────────────────
REQUIRED = [
    'AZURE_OPENAI_ENDPOINT',
    'AZURE_OPENAI_API_KEY',
    'AZURE_OPENAI_REASONING_MODEL',
    'AZURE_OPENAI_EMB_DEPLOYMENT',
]
missing = [k for k in REQUIRED if not os.getenv(k)]
if missing:
    print(f'❌ Missing environment variables: {missing}')
    print('   Check your .env file before continuing.')
else:
    print('✅ All required environment variables found')
    print(f'   Endpoint: {os.getenv("AZURE_OPENAI_ENDPOINT")}')
    print(f'   Model:    {os.getenv("AZURE_OPENAI_REASONING_MODEL")}')

# ── imports ────────────────────────────────────────────────────────────────
from openai import AzureOpenAI
from memory import AgentMemory, AgentMemoryConfig

print('\n✅ Cell 1 complete — ready to configure your agent')

## Cell 2 — Configure Your Agent

This is where your scenario choice comes in. Fill in the three `# YOUR CHOICE:` sections:

1. **`MY_SCENARIO`** — your letter (A, B, or C)
2. **`USER_ID`** — a unique ID for your fictional user
3. **`AGENT_PERSONA`** — the agent's instructions (a short description of its role and what it should remember)


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# YOUR CHOICE: Paste the block for your scenario here
# ═══════════════════════════════════════════════════════════════════

MY_SCENARIO   = '___'          # Replace with A, B, or C
USER_ID       = '___'          # Replace with the user ID for your scenario
AGENT_PERSONA = '___'          # Replace with the agent instructions for your scenario

# ═══════════════════════════════════════════════════════════════════
# Validation — do not change below this line
# ═══════════════════════════════════════════════════════════════════
assert MY_SCENARIO in ('A', 'B', 'C'), 'MY_SCENARIO must be A, B, or C'
assert USER_ID != '___', 'Fill in USER_ID'
assert AGENT_PERSONA != '___', 'Fill in AGENT_PERSONA'
print(f'✅ Scenario {MY_SCENARIO} selected')
print(f'   User ID:  {USER_ID}')
print(f'   Persona:  {AGENT_PERSONA[:80]}...')

## Cell 3 — Tune Memory Configuration

Use this cell to control how your agent stores and recalls memory. Focus on these key points:

- Set `buffer_size` to control when older turns are compressed into summaries.
- Set `active_turns` to decide how many recent turns stay verbatim.
- Keep `auto_enrich_context=True` for automatic recall when trigger words appear.
- Add useful scenario keywords in `enrichment_trigger_keywords` so recall happens at the right time.
- Use `longterm_synthesis_frequency` to control how often long-term profile updates run.

Start with the default values, run the notebook once, then tune one parameter at a time to observe behavior changes.

In [ ]:
# ── Azure OpenAI client ────────────────────────────────────────────────────
openai_client = AzureOpenAI(
    azure_endpoint=os.getenv('AZURE_OPENAI_ENDPOINT'),
    api_key=os.getenv('AZURE_OPENAI_API_KEY'),
    api_version=os.getenv('AZURE_OPENAI_API_VERSION', '2025-04-01-preview'),
)
print('✅ Azure OpenAI client created')

# ── Memory configuration ───────────────────────────────────────────────────
# These are tuning parameters. The values below work well for a short lab
# session. You may adjust them and re-run after getting a working result.

config = AgentMemoryConfig(
    buffer_size=4,                  # compress after 4 turns
    active_turns=2,                 # keep last 2 turns verbatim
    auto_enrich_context=True,       # auto-search when trigger keywords appear
    enrichment_trigger_keywords=[   # words that trigger auto memory search
        'remember', 'recall',
        'last time', 'before', 'previously',
        'history', 'earlier', 'mentioned',
        # Add your scenario-specific keywords below:
        # e.g. for IT: 'issue', 'problem', 'error'
        # e.g. for Travel: 'trip', 'prefer', 'dietary'
        # e.g. for Learning: 'score', 'studied', 'covered'
    ],
    longterm_synthesis_frequency=1, # update long-term profile after every session
)

# ── DB path for SQLite (used in Session 1 and 2) ──────────────────────────
DB_PATH = str(project_root / f'capstone_{MY_SCENARIO.lower()}_{USER_ID}.db')
print(f'\n✅ Memory config created')
print(f'   DB path:              {DB_PATH}')
print(f'   Buffer size:          {config.buffer_size} turns before compression')
print(f'   Active turns kept:    {config.active_turns}')
print(f'   Auto-enrich:          {config.auto_enrich_context}')
print(f'   Synthesis frequency:  every {config.longterm_synthesis_frequency} session(s)')

## Cell 4 — Build the Simple Agent

This is a lightweight agent class that wraps the Azure OpenAI chat API. It uses the `AGENT_PERSONA` you defined in Cell 2 as its system prompt.

**You do not need to change this cell.** The agent receives memory context through the `{memory_context}` placeholder in your persona string.

In [ ]:
class MyAgent:
    """
    A simple Azure OpenAI agent that accepts a memory context
    and responds to user messages.
    """

    def __init__(self, client: AzureOpenAI, persona_template: str):
        self.client = client
        self.persona_template = persona_template
        self.model = os.getenv('AZURE_OPENAI_REASONING_MODEL', 'gpt-4o')
        self.messages = []

    def set_memory_context(self, context: str):
        """Inject memory context into the system prompt."""
        memory_section = (
            f'\n\n--- MEMORY CONTEXT ---\n{context}\n---'
            if context and context.strip()
            else ''
        )
        system_content = self.persona_template.format(memory_context=memory_section)
        # reset or update the system message
        if self.messages and self.messages[0]['role'] == 'system':
            self.messages[0]['content'] = system_content
        else:
            self.messages.insert(0, {'role': 'system', 'content': system_content})

    def chat(self, user_message: str) -> str:
        """Send a user message and return the agent response."""
        self.messages.append({'role': 'user', 'content': user_message})
        response = self.client.chat.completions.create(
            model=self.model,
            messages=self.messages,
            max_completion_tokens=400,
        )
        reply = response.choices[0].message.content
        self.messages.append({'role': 'assistant', 'content': reply})
        return reply

    def reset(self):
        """Clear conversation history (keeps system prompt)."""
        self.messages = [m for m in self.messages if m['role'] == 'system']


agent = MyAgent(client=openai_client, persona_template=AGENT_PERSONA)
print('✅ MyAgent created')
print(f'   Model: {agent.model}')

## Cell 5 — Session 1: Establish Facts

Session 1 builds the foundation of your agent's memory. You will write a short conversation that **establishes 3–4 memorable facts** about your fictional user.

**Why these specific facts matter:** whatever you write in Session 1 is what your agent should recall in Session 2 — *without you repeating it*. Choose facts that are specific and testable.

### Instructions
1. Read the example conversation for your scenario.
2. Either use the example as-is or replace the turns with your own.
3. Make sure turns 1–3 establish clear, memorable facts. Turn 4 is optional.

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# YOUR CHOICE: Paste your scenario's example SESSION_1_TURNS here
# (or replace with your own turns — but keep 4–6 turns)
# ═══════════════════════════════════════════════════════════════════

SESSION_1_TURNS = [
    # Replace these placeholder tuples with your scenario's turns:
    ('user', 'REPLACE ME — turn 1 user message'),
    ('assistant', 'REPLACE ME — turn 1 agent reply'),
    ('user', 'REPLACE ME — turn 2 user message'),
    ('assistant', 'REPLACE ME — turn 2 agent reply'),
    ('user', 'REPLACE ME — turn 3 user message'),
    ('assistant', 'REPLACE ME — turn 3 agent reply'),
]

# ═══════════════════════════════════════════════════════════════════
# RUN Session 1 — do not change below this line
# ═══════════════════════════════════════════════════════════════════
print(f'Running Session 1 for user: {USER_ID}')
print('=' * 60)

async def run_session_1():
    async with AgentMemory(
        user_id=USER_ID,
        openai_client=openai_client,
        db_path=DB_PATH,
        config=config,
    ) as memory:
        await memory.start_session()
        context = await memory.get_context()
        print(f'Memory context at start: {len(context)} chars')
        if context.strip():
            print(f'Prior memory loaded:\n{context[:300]}...')
        else:
            print('No prior memory — this is the first session.')

        agent.reset()
        agent.set_memory_context(context)

        for role, content in SESSION_1_TURNS:
            print(f'\n[{role.upper()}]: {content}')
            if role == 'user':
                reply = agent.chat(content)
                print(f'[AGENT]: {reply}')
                await memory.add_turn(content, reply)

        print('\n' + '─' * 60)
        print('Session 1 ending — triggering reflection...')
        final_ctx = await memory.get_context()
        print(f'Final context: {len(final_ctx)} chars')
        insights = await memory.get_insights(limit=10)
        print(f'Insights extracted: {len(insights)}')
        for i, ins in enumerate(insights, 1):
            print(f'  {i}. {ins}')

await run_session_1()
print('\n✅ Session 1 complete')

## Cell 6 — Build an Explicit Memory Retrieval Tool

In Exercise 2 you watched a pre-built agent (the Medical Assistant) call an explicit `search_memory` tool to recall the penicillin allergy. Here you build that pattern yourself.

An **explicit memory retrieval tool** is a Python function your agent can call by decision — when it recognises that a question needs information from past sessions. It is different from automatic context injection (`auto_enrich_context=True`) because:

- **Automatic:** memory is always injected before the model responds, whether or not it is relevant.
- **Explicit (this cell):** the agent decides when to call the tool based on the conversation. Every lookup is visible and auditable.

**Fill in the `# YOUR CHOICE:` section** with a relevant search query for your scenario:

| Scenario | Example test query |
|---|---|
| A — IT Help Desk | `'What device and OS does this user have? Any known issues?'` |
| B — Travel Planning | `'What are this user\'s travel preferences, dietary restrictions, and past trips?'` |
| C — Learning Coach | `'What topics has this user covered and what is their learning style?'` |

In [ ]:
# ── Explicit memory retrieval tool ─────────────────────────────────────────
# This function is the equivalent of the search_memory tool from Exercise 2.
# It searches across stored interactions AND extracted insights.

async def recall_user_history(memory: AgentMemory, query: str) -> str:
    """
    Retrieve relevant history about the user from long-term memory.
    Call this when the user references something from a previous session,
    or when you need prior context before making a recommendation.
    """
    results = await memory.search(
        query=query,
        top_k=3,
        search_interactions=True,
        search_insights=True,
    )
    if not results:
        return 'No relevant history found in memory.'
    return f'Relevant history from memory:\n{results}'


# ── Test the tool before using it in Session 2 ────────────────────────────
# ═══════════════════════════════════════════════════════════════════
# YOUR CHOICE: Replace the test query with one relevant to your scenario
# (see the table above)
# ═══════════════════════════════════════════════════════════════════

TEST_QUERY = 'REPLACE ME — enter a search query relevant to your scenario'

# ═══════════════════════════════════════════════════════════════════

async def test_retrieval_tool():
    async with AgentMemory(
        user_id=USER_ID,
        openai_client=openai_client,
        db_path=DB_PATH,
        config=config,
    ) as memory:
        await memory.start_session()
        print(f'Testing retrieval tool with query:')
        print(f'  "{TEST_QUERY}"')
        print('─' * 60)
        result = await recall_user_history(memory, TEST_QUERY)
        print(result)
        return result

result = await test_retrieval_tool()
print('\n✅ Retrieval tool test complete')
print('   If the result above contains facts from Session 1, your tool is working.')

## Cell 7 — Session 2: Prove Cross-Session Recall

Session 2 starts as a **completely fresh process** — a new `AgentMemory` instance with an empty active buffer. Yet it should immediately know everything from Session 1, because the server loads prior insights and session summaries at session start.

Your Session 2 conversation must include:
- At least one question that **requires a fact from Session 1** to answer correctly (without the user re-stating it).
- At least one **explicit memory tool call** — a moment where you call `recall_user_history()` before the agent responds.

**The verification test:** if the agent's response to the Session 1-dependent question correctly uses the fact the user never repeated, cross-session recall is proven.

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# YOUR CHOICE: Paste your scenario's SESSION_2_TURNS and RECALL_BEFORE_TURN
# ═══════════════════════════════════════════════════════════════════

SESSION_2_TURNS = [
    # Replace with your scenario's Session 2 turns
    ('user', 'REPLACE ME — Session 2 turn 1'),
    ('user', 'REPLACE ME — Session 2 turn 2'),
]

# Which turn index (0-based) should trigger the explicit recall tool?
RECALL_BEFORE_TURN = 0

# ═══════════════════════════════════════════════════════════════════
# RUN Session 2 — do not change below this line
# ═══════════════════════════════════════════════════════════════════
print(f'Running Session 2 for user: {USER_ID}')
print('=' * 60)

async def run_session_2():
    async with AgentMemory(
        user_id=USER_ID,
        openai_client=openai_client,
        db_path=DB_PATH,
        config=config,
    ) as memory:
        await memory.start_session()

        context = await memory.get_context()
        print(f'Context loaded at Session 2 start: {len(context)} chars')
        if not context.strip():
            print('⚠️  Context is empty — Session 1 may not have stored data correctly. Re-run Cell 5.')
        else:
            print('✅ Prior memory loaded — cross-session recall is active')
            print(f'Preview: {context[:400]}...')

        agent.reset()
        agent.set_memory_context(context)

        for i, (role, content) in enumerate(SESSION_2_TURNS):
            print(f'\n[{role.upper()}]: {content}')

            if i == RECALL_BEFORE_TURN:
                # ── Explicit retrieval tool call ──────────────────────────
                print(f'\n  🔍 [Explicit memory tool called]')
                retrieved = await recall_user_history(memory, content)
                print(f'  Retrieved: {retrieved[:300]}...')
                # Inject the retrieval result into the agent's context
                enriched_message = (
                    f'{content}\n\n'
                    f'[Memory retrieved for this question: {retrieved}]'
                )
                reply = agent.chat(enriched_message)
                # Store the original message + reply (not the enriched one)
                await memory.add_turn(content, reply)
            else:
                reply = agent.chat(content)
                await memory.add_turn(content, reply)

            print(f'[AGENT]: {reply}')

        print('\n' + '─' * 60)
        print('Session 2 ending...')
        sessions = await memory.get_sessions()
        insights = await memory.get_insights(limit=10)
        print(f'Total sessions recorded: {len(sessions)}')
        print(f'Total insights now:      {len(insights)}')
        for i, ins in enumerate(insights, 1):
            print(f'  {i}. {ins}')

await run_session_2()
print('\n✅ Session 2 complete')

## Cell 8 — Verify and Reflect

This final cell runs three verification checks and prints a summary of everything your agent built.

**What it checks:**
1. Were turns stored in the database?
2. Were insights extracted?
3. Does Session 2 load a non-empty context (proving cross-session recall)?

**You do not need to change anything in this cell.**

In [ ]:
async def verify_and_summarise():
    async with AgentMemory(
        user_id=USER_ID,
        openai_client=openai_client,
        db_path=DB_PATH,
        config=config,
    ) as memory:
        await memory.start_session()

        context = await memory.get_context()
        sessions = await memory.get_sessions()
        insights = await memory.get_insights(limit=20)

        print('╔══════════════════════════════════════════════════════════╗')
        print('║         EXERCISE 6 CAPSTONE — VERIFICATION REPORT       ║')
        print('╚══════════════════════════════════════════════════════════╝')
        print(f'  Scenario:    {MY_SCENARIO}')
        print(f'  User ID:     {USER_ID}')
        print()

        # Check 1: Sessions recorded
        sessions_ok = len(sessions) >= 2
        print(f'  ✅ Sessions recorded: {len(sessions)}'  if sessions_ok
              else f'  ❌ Sessions recorded: {len(sessions)} (expected ≥ 2 — re-run Cells 5 and 7)')

        # Check 2: Insights extracted
        insights_ok = len(insights) >= 1
        print(f'  ✅ Insights extracted: {len(insights)}'  if insights_ok
              else f'  ❌ Insights extracted: {len(insights)} (expected ≥ 1 — check your turns in Cell 5)')

        # Check 3: Cross-session context
        context_ok = len(context) > 50
        print(f'  ✅ Cross-session context: {len(context)} chars'  if context_ok
              else f'  ❌ Cross-session context: {len(context)} chars (too small — session end may not have stored data)')

        print()
        print('  Insights summary:')
        for i, ins in enumerate(insights, 1):
            print(f'    {i}. {str(ins)[:120]}')

        print()
        print('  Session summaries:')
        for s in sessions:
            print(f'    - {str(s)[:120]}')

        print()
        print('  Context preview (first 500 chars):')
        print(f'  {context[:500]}')

        all_ok = sessions_ok and insights_ok and context_ok
        print()
        print('══════════════════════════════════════════════════════════')
        if all_ok:
            print('  🎉 ALL CHECKS PASSED — your memory-enabled agent works!')
        else:
            print('  ⚠️  Some checks failed — see messages above for guidance.')
        print('══════════════════════════════════════════════════════════')

await verify_and_summarise()